# patentrag compute pipeline (GPU)

Runs indexes, graph, tests, tuning on validation queries, the test evaluation, ablations, robustness, the agent's claim-chart reports and the independent audit.

1. Runtime > Change runtime type > T4 GPU.
2. Build the bundle locally with `python scripts/make_colab_bundle.py` and upload `colab/patentrag_bundle.zip` to `/content` (Files panel).
3. Run the cells in order. The pipeline runs in the background; re-run the status cell to follow it (several hours on a T4).
   If a step is killed, run `bash scripts/pipeline.sh agent` in the same VM: saved agent runs are skipped. A new VM starts empty, so export finished files early.
4. Copy the results out with the export cell and `scripts/import_results.py`.


In [ ]:
import subprocess, torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print(subprocess.run('nproc; free -g | head -2', shell=True, capture_output=True, text=True).stdout)


In [ ]:
# setup and launch
import zipfile, subprocess
zipfile.ZipFile('/content/patentrag_bundle.zip').extractall('/content/patentrag')
subprocess.Popen('bash scripts/colab_setup.sh > /content/setup.log 2>&1 && bash scripts/pipeline.sh > /content/pipeline.log 2>&1', shell=True, cwd='/content/patentrag')
print('started')


In [ ]:
# status
import subprocess
print(subprocess.run('tail -3 /content/setup.log; tail -30 /content/patentrag/logs/progress.txt; tail -3 /content/pipeline.log', shell=True, capture_output=True, text=True).stdout)


In [ ]:
# export: compact, checksummed text to copy into a local file for scripts/import_results.py
import subprocess
files = 'tuned_params_full.json eval_full_test.json eval_full_leak_demo.json eval_full_leak_demo_default.json ablations_full_test.json robustness_full.json faithfulness_full.json faithfulness_audit_full.json nli_threshold.json failure_analysis_full.json failure_analysis_full.md e2e_smoke_full.json tables_full.md'
print(subprocess.run('python scripts/export_results.py ' + files, shell=True, cwd='/content/patentrag', capture_output=True, text=True).stdout)
